> **Notebook d'approfondissement** — à lire si `.query()` + sélection de colonnes ne vous suffit plus,
> ou si vous comprenez un bug de sélection que vous n'arrivez pas à expliquer.

# Subtilités de `.loc` et `.iloc`

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_penguins

df = load_accidents()
penguins = load_penguins()

## 1. La distinction fondamentale : label vs position

| | `.loc[]` | `.iloc[]` |
|---|---|---|
| Principe | Sélection par **label** (valeur de l'index) | Sélection par **position entière** (0, 1, 2…) |
| Slicing | Borne droite **incluse** | Borne droite **exclue** (comme Python) |
| Risque | Lent si l'index n'est pas trié | Fragile si les lignes sont réordonnées |

Sur un DataFrame avec un RangeIndex (0, 1, 2…), label == position, et les deux semblent identiques.
C'est le piège : dès qu'on filtre ou trie, les positions changent mais les labels restent.

In [ ]:
# Créer un exemple minimal pour visualiser
small = pd.DataFrame(
    {"val": [10, 20, 30, 40, 50]},
    index=[0, 1, 2, 3, 4],
)

# Sur un RangeIndex, loc et iloc semblent identiques
print("loc[1:3] :", small.loc[1:3, "val"].tolist())   # 3 éléments — INCLUS
print("iloc[1:3]:", small.iloc[1:3]["val"].tolist())  # 2 éléments — EXCLUS

In [ ]:
# Après un filtre, les labels (index) ne correspondent plus aux positions
filtre = small[small["val"] > 15]  # lignes 1, 2, 3, 4 — index inchangé
print("Index après filtre :", filtre.index.tolist())

print("\nfiltre.loc[1]  :", filtre.loc[1, "val"])   # label 1 → 20
print("filtre.iloc[1] :", filtre.iloc[1]["val"])   # position 1 → 30 (le 2e élément du résultat)

**Leçon** : après un `.query()`, un `.sort_values()`, ou un `.groupby()`, l'index ne repart pas de 0.
`.iloc[0]` n'est pas "la première ligne du dataset original" — c'est "la première ligne de cet objet".
`.loc[0]` cherche le label `0`, qui peut ne pas exister après un filtre.

## 2. `reset_index()` et `set_index()`

Après un filtre ou un tri, l'index est "troué" (0, 2, 5, 9…).
`reset_index()` remet un RangeIndex propre.

In [ ]:
accidents_paris = df.query("dep == '75'").reset_index(drop=True)
print("Index avant reset :", df.query("dep == '75'").index[:5].tolist())
print("Index après reset :", accidents_paris.index[:5].tolist())

In [ ]:
# set_index() : utiliser une colonne comme index
# Utile quand on veut accéder aux lignes par un identifiant métier
df_indexed = df.set_index("Num_Acc")
df_indexed.head(3)

In [ ]:
# Maintenant .loc[] cherche par Num_Acc
num = df["Num_Acc"].iloc[0]
df_indexed.loc[num, ["dep", "mois", "lum"]]

> **Note** : un index non-entier rend `.iloc[]` indispensable pour l'accès positionnel,
> puisque `.loc[]` cherche maintenant par `Num_Acc`.

## 3. `.loc[]` pour modifier des valeurs — la seule façon correcte

Modifier une valeur dans une DataFrame se fait **uniquement avec `.loc[]`**.
L'indexation chaînée `df[condition]["col"] = valeur` peut modifier une copie
et ne pas affecter le DataFrame original — comportement dépendant de l'interne pandas.

Avant d'exécuter — **que va-t-il se passer avec la version chaînée ?**

In [ ]:
sample = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

# Version chaînée — comportement non garanti (SettingWithCopyWarning en pandas < 2.0)
sample[sample["a"] > 1]["b"] = 999
sample  # b est-il modifié ?

In [ ]:
# Version correcte avec .loc[]
sample.loc[sample["a"] > 1, "b"] = 999
sample

## 4. `.at[]` et `.iat[]` — accès scalaire rapide

Pour lire ou écrire **une seule valeur** (un scalaire), `.at[]` et `.iat[]` sont
significativement plus rapides que `.loc[]` et `.iloc[]`.
La raison : ils ne font pas les vérifications d'alignement ni la création d'objets intermédiaires.

In [ ]:
# .at[label_ligne, nom_colonne]
print(df.at[0, "dep"])       # label 0, colonne dep

# .iat[position_ligne, position_colonne]
print(df.iat[0, 6])          # ligne 0, 7e colonne

In [ ]:
%%timeit
df.loc[0, "dep"]

In [ ]:
%%timeit
df.at[0, "dep"]

En pratique : `.at[]` / `.iat[]` sont utiles dans les rares boucles sur les lignes
où la performance compte, et pour les fonctions de validation qui lisent une cellule précise.
Dans une chaîne de transformation normale, `.loc[]` est largement suffisant.

## 5. `idxmin()` et `idxmax()` — trouver le label, pas la valeur

In [ ]:
# Quel département a la vitesse maximale autorisée la plus élevée sur ses voies ?
# vma = vitesse maximale autorisée (colonne de la table LIEUX)
idx_max_vma = df["vma"].idxmax()  # retourne le label de la ligne, pas la valeur

print("Label de la ligne :", idx_max_vma)
print("Valeur de vma     :", df.at[idx_max_vma, "vma"])
df.loc[idx_max_vma, ["dep", "mois", "vma", "catr"]]

## 6. MultiIndex — sélection hiérarchique

Un MultiIndex est un index à plusieurs niveaux. Il apparaît naturellement après un `groupby`
ou un `pivot_table`, et peut aussi être construit explicitement.

In [ ]:
# Construire un MultiIndex via groupby
multi = (
    df
    .groupby(["dep", "mois"])["Num_Acc"]
    .count()
    .rename("nb_accidents")
)
multi.head(10)

In [ ]:
# Sélectionner le niveau supérieur (dep) avec .loc
multi.loc["75"]  # tous les mois pour Paris

In [ ]:
# Sélectionner un couple (dep, mois) précis
multi.loc[("75", 6)]  # Paris, juin

In [ ]:
# pd.IndexSlice : syntaxe lisible pour les sélections complexes
idx = pd.IndexSlice

# Tous les départements, uniquement les mois d'été (6, 7, 8)
multi.loc[idx[:, [6, 7, 8]]].head(10)

In [ ]:
# Revenir à un index plat avec reset_index()
multi.reset_index().head(5)

## 7. Copy vs View — pourquoi ça compte

Une **view** partage la mémoire avec le DataFrame parent : modifier la view modifie l'original.
Une **copy** est indépendante.

pandas ne garantit pas lequel vous obtenez avec une sélection simple — c'est dépendant
de la structure interne. Depuis pandas 2.0, le **Copy-on-Write** (CoW) est activé par défaut :
toute modification crée une copie, éliminant les bugs de view silencieux.
Ce sujet est détaillé dans `pandas_traps_v2.ipynb`.

In [ ]:
# Vérifier si CoW est activé
print(pd.options.mode.copy_on_write)  # True en pandas >= 2.0

In [ ]:
# Avec CoW, cette opération ne modifie plus df — elle crée une copie implicite
subset = df[["dep", "mois"]]
subset.iloc[0, 0] = "XX"  # modifie subset, pas df

print("df intact :", df.iloc[0, df.columns.get_loc("dep")])

## 8. Accès par ligne avec `.xs()` — cross-section

`.xs()` (*cross-section*) est une syntaxe alternative pour sélectionner
un niveau d'un MultiIndex ou une ligne spécifique. Moins courant mais parfois plus lisible
que `.loc[]` sur des structures hiérarchiques.

In [ ]:
# Équivalent à multi.loc["75"] mais plus explicite sur le niveau
multi.xs("75", level="dep")

In [ ]:
# Sélectionner un mois spécifique sur tous les départements (niveau 1)
multi.xs(6, level="mois").nlargest(10)

## Récapitulatif

| Besoin | Outil |
|---|---|
| Filtrer des lignes sur une condition | `.query()` (défaut) ou masque booléen |
| Sélectionner par label d'index | `.loc[label]` |
| Sélectionner par position entière | `.iloc[n]` |
| Lire/écrire une seule cellule (perf) | `.at[]` / `.iat[]` |
| Trouver la ligne du min/max | `.idxmin()` / `.idxmax()` |
| MultiIndex : niveau supérieur | `.loc["val"]` ou `.xs("val", level=...)` |
| MultiIndex : plage complexe | `pd.IndexSlice` + `.loc[]` |
| Remettre un index propre | `.reset_index(drop=True)` |
| Utiliser une colonne comme index | `.set_index("col")` |